In [2]:
import pandas as pd
import numpy as np

### Reading CSV File

In [ ]:
df = pd.read_csv("DTO_cleaned5.csv")
df.head(10)

# Make train test split of 80,10,10 using stratified sampling


AttributeError: module 'scipy' has no attribute '_lib'

In [9]:
import sklearn
from sklearn.model_selection import train_test_split
train, test = train_test_split(df, test_size=0.2)
test, val = train_test_split(test, test_size=0.5)
train.shape,val.shape,test.shape

AttributeError: module 'scipy' has no attribute '_lib'

### Making triples

In [9]:
triples = df["subject"] + " " + df["predicate"] + " " + df["object"]    
triples.head

0    http://www.drugtargetontology.org/dto/DTO_6200...
1    http://www.drugtargetontology.org/dto/DTO_6200...
2    http://www.drugtargetontology.org/dto/DTO_6200...
3    http://www.drugtargetontology.org/dto/DTO_6200...
4    http://www.drugtargetontology.org/dto/DTO_6200...
5    http://www.drugtargetontology.org/dto/DTO_6200...
6    http://www.drugtargetontology.org/dto/DTO_0310...
7    http://www.drugtargetontology.org/dto/DTO_0310...
8    http://www.drugtargetontology.org/dto/DTO_0310...
9    http://www.drugtargetontology.org/dto/DTO_0310...
dtype: object

### Generating Graph Embedding using DistMult

In [ ]:
from models.base_model import BaseModel
from models.param import LookupParameter
from utils.math_utils import *


class DistMult(BaseModel):
    def __init__(self, **kwargs):
        self.n_entity = kwargs.pop('n_entity')
        self.n_relation = kwargs.pop('n_relation')
        self.dim = kwargs.pop('dim')
        self.margin = kwargs.pop('margin')
        mode = kwargs.pop('mode', 'pairwise')
        if mode == 'pairwise':
            self.compute_gradients = self._pairwisegrads
        elif mode == 'single':
            self.compute_gradients = self._singlegrads
        else:
            raise NotImplementedError

        self.params = {'e': LookupParameter(name='e', shape=(self.n_entity, self.dim)),
                       'r': LookupParameter(name='r', shape=(self.n_relation, self.dim))}

    def _pairwisegrads(self, pos_samples, neg_samples):
        assert pos_samples.shape == neg_samples.shape
        self.prepare()
        p_scores = self.cal_triplet_scores(pos_samples)
        n_scores = self.cal_triplet_scores(neg_samples)

        loss = max_margin(p_scores, n_scores)
        idxs = np.where(loss > 0)[0]
        if len(idxs) != 0:
            # TODO: inefficient calculation
            pos_subs, pos_rels, pos_objs = pos_samples[idxs, 0], pos_samples[idxs, 1], pos_samples[idxs, 2]
            neg_subs, neg_rels, neg_objs = neg_samples[idxs, 0], neg_samples[idxs, 1], neg_samples[idxs, 2]

            p_s_embs = self.pick_ent(pos_subs)
            p_r_embs = self.pick_rel(pos_rels)
            p_o_embs = self.pick_ent(pos_objs)
            n_s_embs = self.pick_ent(neg_subs)
            n_r_embs = self.pick_rel(neg_rels)
            n_o_embs = self.pick_ent(neg_objs)

            _batchsize = len(pos_subs)

            p_s_grads = - (p_r_embs * p_o_embs)
            p_r_grads = - (p_s_embs * p_o_embs)
            p_o_grads = - (p_s_embs * p_r_embs)
            n_s_grads = n_r_embs * n_o_embs
            n_r_grads = n_s_embs * n_o_embs
            n_o_grads = n_s_embs * n_r_embs

            for idx in range(_batchsize):
                self.params['e'].add_grad(pos_subs[idx], p_s_grads[idx])
                self.params['r'].add_grad(pos_rels[idx], p_r_grads[idx])
                self.params['e'].add_grad(pos_objs[idx], p_o_grads[idx])
                self.params['e'].add_grad(neg_subs[idx], n_s_grads[idx])
                self.params['r'].add_grad(neg_rels[idx], n_r_grads[idx])
                self.params['e'].add_grad(neg_objs[idx], n_o_grads[idx])

        else:
            pass

        self.params['e'].finalize()
        self.params['r'].finalize()

        return loss.mean()

    def _singlegrads(self, samples, ys):
        raise NotImplementedError('Only pairwise setting is available')

    def _composite(self, sub_emb, rel_emb):
        return sub_emb * rel_emb

    def _cal_similarity(self, query, obj_emb):
        return np.sum(query * obj_emb, axis=1)

    def cal_scores(self, subs, rels):
        sub_emb = self.pick_ent(subs)
        rel_emb = self.pick_rel(rels)
        qs = self._composite(sub_emb, rel_emb)
        score_mat = qs.dot(self.params['e'].data.T)
        return score_mat

    # TODO: this procedure is the same as cal_scores
    def cal_scores_inv(self, rels, objs):
        obj_emb = self.pick_ent(objs)
        rel_emb = self.pick_rel(rels)
        qs_inv = obj_emb * rel_emb
        score_mat = qs_inv.dot(self.params['e'].data.T)
        return score_mat

    def cal_triplet_scores(self, samples):
        subs, rels, objs = samples[:, 0], samples[:, 1], samples[:, 2]
        sub_emb = self.pick_ent(subs)
        rel_emb = self.pick_rel(rels)
        obj_emb = self.pick_ent(objs)
        qs = self._composite(sub_emb, rel_emb)
        return self._cal_similarity(qs, obj_emb)

    def pick_ent(self, ents):
        return self.params['e'].data[ents]

    def pick_rel(self, rels):
        return self.params['r'].data[rels]

In [11]:
! pip install torch_geometric

  Using cached requests-2.32.3-py3-none-any.whl.metadata (4.6 kB)
  Using cached idna-3.10-py3-none-any.whl.metadata (10 kB)
   ---------------------------------------- 0.0/1.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.1 MB ? eta -:--:--
   --------- ------------------------------ 0.3/1.1 MB ? eta -:--:--
   --------- ------------------------------ 0.3/1.1 MB ? eta -:--:--
   --------- ------------------------------ 0.3/1.1 MB ? eta -:--:--
   --------- ------------------------------ 0.3/1.1 MB ? eta -:--:--
   --------- ------------------------------ 0.3/1.1 MB ? eta -:--:--
   --------- ------------------------------ 0.3/1.1 MB ? eta -:--:--
   ------------------ --------------------- 0.5/1.1 MB 207.2 kB/s eta 0:00:03
   ------------------ --------------------- 0.5/1.1 MB 207.2 kB/s eta 0:00:03
   ------------------ -------

In [ ]:
import torch
from torch_geometric.nn import Node2Vec
import networkx as nx

In [ ]:
# Read csv file line by line and store in text file

import csv
import os

def read_csv(file):
    with open(file,'r') as file:
        reader = csv.reader(file)
        for row in reader:

            with open("")

In [3]:
import torch

In [9]:
from pykeen.triples import TriplesFactory

In [6]:
model = torch.load("trained_model.pkl",map_location=torch.device('cpu'))
model

C:\Users\Shivesh\AppData\Local\Temp\ipykernel_35960\3929901308.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model = torch.load("trained_model.pkl",map_location=torch.

TransE(
  (loss): MarginRankingLoss(
    (margin_activation): ReLU()
  )
  (interaction): TransEInteraction()
  (entity_representations): ModuleList(
    (0): Embedding(
      (_embeddings): Embedding(83717, 50)
    )
  )
  (relation_representations): ModuleList(
    (0): Embedding(
      (_embeddings): Embedding(1024, 50)
    )
  )
  (weight_regularizers): ModuleList()
)

In [1]:
! pip install pykeen

In [11]:
triples_factory = TriplesFactory.from_path_binary("results\\training_triples")
print(triples_factory.num_entities)
print(triples_factory.num_relations)

NotImplementedError: cannot instantiate 'PosixPath' on your system